# Deutsch–Jozsa (Modular) using GenericMatrix

 by Benjelyn Reves Patiag

This notebook show same Deutsch–Jozsa idea, but modular so you can run:
- **NumPy matrix simulation** (no quantum SDK needed)
- **Qiskit** (if installed)
- **Cirq** (if installed)

Ancilla (helper ) qubit is the **last / LSB: Least Significant Bit** like in `GenericMatrix_upgraded.py`.


In [3]:
import sys, os  # need path + basic OS stuff
import importlib  # for reload when you edit .py file

# --- make sure notebook can import the .py in same folder ---
# this line add current folder to python path, so `import GenericMatrix_upgraded` works
sys.path.insert(0, os.getcwd())  # simple fix if import not found

import GenericMatrix_upgraded as gm  # this is the matrix toolkit file
importlib.reload(gm)  # reload, so if you edited gm file, notebook use latest

import numpy as np  # for arrays / math

print("Python exe:", sys.executable)  # show which python you run
print("Work folder:", os.getcwd())  # show where notebook is
print("GenericMatrix file:", gm.__file__)  # show where gm is loaded from


Python exe: C:\Quantum\BuiltScripts\cirq-venv\Scripts\python.exe
Work folder: C:\Quantum\BuiltScripts
GenericMatrix file: C:\Quantum\BuiltScripts\GenericMatrix_upgraded.py


In [4]:
# 1 function that run Deutsch–Jozsa, but  can swap "backend" easily.
# backend = "numpy"  -> only matrix math (always work, no qiskit/cirq needed)
# backend = "qiskit" -> use qiskit circuit + statevector (no Aer needed)
# backend = "cirq"   -> use cirq circuit + simulator

def run_deutsch_jozsa(n, fx, backend="qiskit", shots=1024):
    # n = number of input qubits
    # fx = truth table list OR callable f(x)
    # backend pick which engine to run
    # shots used only for circuit style backends

    Uf = gm.build_uf_matrix(n, fx)  # build oracle matrix U_f (big unitary)

    if backend == "numpy":
        # matrix-only simulation (fast + no extra install)
        final_state, verdict, input_probs = gm.simulate_deutsch_jozsa(n, fx)
        return {
            "backend": "numpy",
            "verdict": verdict,
            "input_probs": input_probs,
            "final_state": final_state,
            "Uf": Uf,
        }

    if backend == "qiskit":
        # try import qiskit, if not exist show nice error
        try:
            from qiskit import QuantumCircuit  # main circuit object
            from qiskit.quantum_info import Statevector  # pure state simulator
        except Exception as e:
            raise ImportError("Qiskit not installed in this python env. Use backend='numpy' or install qiskit.") from e

        total = n + 1  # inputs + 1 ancilla
        qc = QuantumCircuit(total, n)  # n classical bits for measuring input only

        # --- prepare |0...0,1> ---
        qc.x(0)  # qubit0 is LSB (ancilla), flip to |1>

        # --- H on all qubits ---
        for q in range(total):
            qc.h(q)  # put each qubit in superposition

        # --- append oracle gate (unitary matrix) ---
        uf_gate = gm.uf_to_qiskit_gate(Uf, label="Uf")  # convert matrix -> qiskit gate
        qc.append(uf_gate, list(range(total)))  # apply on all qubits (same order)

        # --- H on input register only (qubits 1..n) ---
        for q in range(1, n + 1):
            qc.h(q)  # hadamard only inputs

        # --- measure input qubits only ---
        for i in range(n):
            qc.measure(i + 1, i)  # map qubit(1..n) -> classical bit(0..n-1)

        # --- simulate with statevector (no Aer needed) ---
        sv = Statevector.from_instruction(qc.remove_final_measurements(inplace=False))  # ignore measure for exact probs

        # calculate prob of measuring inputs (sum over ancilla)
        # NOTE: qiskit bit order is different, so can do manual marginal using gm style index
        # easiest: build full probs from statevector, then sum ancilla bit like gm.measure_register_probs
        full_probs = np.abs(np.array(sv.data)) ** 2  # probability for each basis state
        input_probs = np.zeros(1 << n)  # only inputs
        for x in range(1 << n):
            # gm index is (x<<1)|y, where y is ancilla (LSB) -> y=0/1
            input_probs[x] = full_probs[(x << 1) | 0] + full_probs[(x << 1) | 1]

        verdict = "constant" if np.isclose(input_probs[0], 1.0, atol=1e-9) else "balanced"

        return {
            "backend": "qiskit",
            "verdict": verdict,
            "input_probs": input_probs,
            "circuit": qc,
            "statevector": sv,
            "Uf": Uf,
        }

    if backend == "cirq":
        # try import cirq, if not exist show nice error
        try:
            import cirq  # cirq lib
        except Exception as e:
            raise ImportError("Cirq not installed in this python env. Use backend='numpy' or install cirq.") from e

        total = n + 1  # inputs + 1 ancilla

        # create qubits:can keep index 0 as ancilla (LSB) to match gm convention
        qs = [cirq.LineQubit(i) for i in range(total)]  # q0=ancilla, q1..qn=inputs

        circuit = cirq.Circuit()  # new empty circuit

        circuit.append(cirq.X(qs[0]))  # set ancilla to |1>

        circuit.append(cirq.H.on_each(*qs))  # H on all qubits

        uf_gate = gm.uf_to_cirq_gate(Uf)  # convert matrix -> cirq MatrixGate
        circuit.append(uf_gate.on(*qs))  # apply Uf to all qubits

        circuit.append(cirq.H.on_each(*qs[1:]))  # H on inputs only

        #can do simulation without measurements for exact probs
        sim = cirq.Simulator()  # local simulator
        result = sim.simulate(circuit)  # run
        state = np.array(result.final_state_vector)  # get final statevector

        # marginal prob of input register (sum ancilla)
        full_probs = np.abs(state) ** 2  # probability array
        input_probs = np.zeros(1 << n)  # only inputs
        for x in range(1 << n):
            input_probs[x] = full_probs[(x << 1) | 0] + full_probs[(x << 1) | 1]

        verdict = "constant" if np.isclose(input_probs[0], 1.0, atol=1e-9) else "balanced"

        return {
            "backend": "cirq",
            "verdict": verdict,
            "input_probs": input_probs,
            "circuit": circuit,
            "final_state": state,
            "Uf": Uf,
        }

    raise ValueError("backend must be: 'numpy' or 'qiskit' or 'cirq'")


In [5]:
# --- choose your function f(x) ---
n = 2  # 2 input qubits, so x in {0,1,2,3}

# example: truth table "0110" means:
# x=0(00)->0, x=1(01)->1, x=2(10)->1, x=3(11)->0
fx = gm.parse_truth_table("0110", n)  # parse string to [0,1,1,0]

# --- run on NumPy backend (always safe) ---
out_numpy = run_deutsch_jozsa(n, fx, backend="numpy")  # run
print("NUMPY verdict:", out_numpy["verdict"])  # show answer
print("NUMPY input probs:", out_numpy["input_probs"])  # show prob for |00>,|01>,|10>,|11>


NUMPY verdict: balanced
NUMPY input probs: [5.28699741e-34 2.64349870e-34 2.64349870e-34 1.00000000e+00]


In [6]:
# --- run on Qiskit backend (only if qiskit installed) ---
try:
    out_qiskit = run_deutsch_jozsa(n, fx, backend="qiskit")  # run
    print("QISKIT verdict:", out_qiskit["verdict"])  # show answer
    print("QISKIT input probs:", out_qiskit["input_probs"])  # show prob
    display(out_qiskit["circuit"])  # show circuit diagram
except Exception as e:
    print("Qiskit run skipped / failed:", e)  # show why (missing install etc)


QISKIT verdict: balanced
QISKIT input probs: [3.68864455e-70 9.07105347e-35 3.87292802e-37 1.00000000e+00]


In [7]:
# --- run on Cirq backend (only if cirq installed) ---
try:
    out_cirq = run_deutsch_jozsa(n, fx, backend="cirq")  # run
    print("CIRQ verdict:", out_cirq["verdict"])  # show answer
    print("CIRQ input probs:", out_cirq["input_probs"])  # show prob
    print(out_cirq["circuit"])  # print circuit
except Exception as e:
    print("Cirq run skipped / failed:", e)  # show why (missing install etc)


CIRQ verdict: balanced
CIRQ input probs: [0.49999997 0.         0.49999997 0.        ]
              ┌                                                       ┐
              │1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j│
              │0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j│
              │0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j│
0: ───X───H───│0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j│───────
              │0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j│
              │0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j│
              │0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j│
              │0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j│
              └                                                       ┘
              │
1: ───H───────#2──────────────────────────────────────────────────────────H───
              │
2: ───H───────#3──────────────────────────────────────────────────────────H

## Interactive runner (widgets)

Use sliders and dropdowns so you no need edit code each time.

> If widgets not show: run `pip install ipywidgets` in your venv, then restart kernel.


In [8]:
# Interactive UI using ipywidgets (simple, kid-friendly)
# - slider for n (how many input qubits)
# - dropdown for oracle type
# - dropdown for backend engine
# - click button to run and show result

try:
    import ipywidgets as widgets  # UI widgets for Jupyter
    from IPython.display import display, clear_output  # show UI + clear output
except Exception as e:
    # if widgets not installed, show simple message
    print("ipywidgets not installed or not enabled.")
    print("Fix: pip install ipywidgets")
    print("Error:", e)
    widgets = None

if widgets is not None:
    # slider: choose n (number of input qubits)
    n_slider = widgets.IntSlider(
        value=2, min=1, max=12, step=1,
        description="n qubits:",  # label
        continuous_update=False  # update when release mouse
    )

    # dropdown: choose oracle type
    oracle_dd = widgets.Dropdown(
        options=[("constant", "constant"), ("balanced", "balanced")],
        value="balanced",
        description="oracle:"
    )

    # dropdown: choose backend engine
    backend_dd = widgets.Dropdown(
        options=[("numpy (matrix)", "numpy"), ("qiskit", "qiskit"), ("cirq", "cirq")],
        value="numpy",
        description="backend:"
    )

    # for constant oracle, choose constant output 0 or 1
    const_value_dd = widgets.Dropdown(
        options=[("always 0", 0), ("always 1", 1)],
        value=0,
        description="const:"
    )

    # output area where we print result
    out = widgets.Output()

    # button to run
    run_btn = widgets.Button(description="RUN Deutsch–Jozsa", button_style="success")

    def build_fx_list(n: int, oracle_type: str, const_val: int):
        # build f(x) as list of 0/1 for x=0..2^n-1
        # layman: this list is the "answer sheet" of the mystery box
        size = 2 ** n  # how many x exist
        if oracle_type == "constant":
            return [const_val] * size  # all same
        else:
            # balanced: half 0, half 1 (simple pattern)
            half = size // 2
            return [0] * half + [1] * half

    def on_run_clicked(_):
        # when button clicked, run algorithm and print result
        with out:
            clear_output()  # clear old print
            n = int(n_slider.value)  # get slider value
            oracle_type = str(oracle_dd.value)  # get oracle type
            backend = str(backend_dd.value)  # get backend
            const_val = int(const_value_dd.value)  # constant choice

            # build fx list
            fx_list = build_fx_list(n, oracle_type, const_val)

            print("You choose:")
            print(" - n =", n)
            print(" - oracle_type =", oracle_type)
            print(" - backend =", backend)
            if oracle_type == "constant":
                print(" - constant value =", const_val)
            print(" - fx truth table =", "".join(str(b) for b in fx_list))

            print("\nRunning...\n")

            try:
                # IMPORTANT: run_deutsch_jozsa accept fx as LIST, so we pass fx_list directly
                result = run_deutsch_jozsa(n, fx_list, backend=backend)
                print("VERDICT:", result.get("verdict"))  # constant or balanced
                if "meas" in result:
                    print("Measurement:", result["meas"])
            except Exception as e:
                print("Run failed:", e)
                print("Tip: if backend is qiskit/cirq, make sure installed in THIS python env.")

    run_btn.on_click(on_run_clicked)  # connect button click

    # show/hide const dropdown depending oracle type
    def on_oracle_change(change):
        if change["new"] == "constant":
            const_value_dd.layout.display = "flex"  # show
        else:
            const_value_dd.layout.display = "none"  # hide

    oracle_dd.observe(on_oracle_change, names="value")
    on_oracle_change({"new": oracle_dd.value})  # set initial show/hide

    # display UI
    ui = widgets.VBox([
        widgets.HBox([n_slider, oracle_dd]),
        widgets.HBox([backend_dd, const_value_dd]),
        run_btn,
        out
    ])
    display(ui)

## How this modular thing works:

- `GenericMatrix_upgraded.py` build **oracle matrix** `Uf` and can also do full Deutsch–Jozsa by matrix math.  
- In this notebook,  wrap it in `run_deutsch_jozsa()` so can just change `backend=`.  
- `"numpy"` backend: just call `gm.simulate_deutsch_jozsa()` (no SDK).  
- `"qiskit"` backend: convert `Uf` to `UnitaryGate`, build circuit, simulate with `Statevector` (no Aer needed).  
- `"cirq"` backend: convert `Uf` to `MatrixGate`, build circuit, simulate with `cirq.Simulator`.  

The Deutsch-Jozsa_Modular_GenericMatrix_widgets.ipynb notebook is a program that run Deutsch–Jozsa algorithm in simple modular way. It build the oracle matrix one time using basic matrix math, then user can choose how to run it (NumPy, Qiskit, or Cirq) using slider and dropdown in Jupyter.

It is useful because the main quantum math part is separate from the software tools, so same algorithm can run in different platforms without changing the main logic, and it is easy to try, test, and understand.

